# Step 1 — Ingest Flight Data

Load raw ADS-B and ADS-C data from the OpenSky dataset.
Organize by flight and order by time.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

# Paths — adjust to your folder layout
BASE_DIR   = Path("../noel_part")
RAW_DIR    = BASE_DIR / "raw_data"
CLEAN_DIR  = BASE_DIR / "cleaned_data_final"

print(f"Raw data folder   : {RAW_DIR}  (exists={RAW_DIR.exists()})")
print(f"Cleaned data folder: {CLEAN_DIR}  (exists={CLEAN_DIR.exists()})")

Raw data folder   : ..\noel_part\raw_data  (exists=False)
Cleaned data folder: ..\noel_part\cleaned_data_final  (exists=True)


## 1a. Run the ETL notebook

The full ingestion and cleaning pipeline is in `etl.ipynb` (your friend's notebook).
Run all cells there first — it produces `cleaned_data_final/` with per-flight folders.

```
noel_part/cleaned_data_final/
└── <step>/
    └── <flight>/
        ├── adsb_before.parquet   ← ADS-B before oceanic gap
        ├── adsc.parquet          ← ADS-C waypoints (ground truth)
        └── adsb_after.parquet    ← ADS-B after oceanic gap
```

In [2]:
import pandas as pd
from pathlib import Path

BASE_DIR  = Path("../noel_part")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"

# Count available flights
steps   = [d for d in CLEAN_DIR.iterdir() if d.is_dir()]
flights = [f for s in steps for f in s.iterdir() if f.is_dir()]

print(f"Steps found   : {len(steps)}")
print(f"Flights found : {len(flights)}")
print()

# Show a sample flight
sample = flights[0]
print(f"Sample flight : {sample.name}")
for f in sorted(sample.iterdir()):
    size = f.stat().st_size // 1024
    print(f"  {f.name:<35} {size:>5} KB")

Steps found   : 24
Flights found : 2149

Sample flight : 20230810_4ba959_073209_092245
  adsb_after.parquet                     24 KB
  adsb_before.parquet                    28 KB
  adsc.parquet                           43 KB


## 1b. Preview a sample flight

In [3]:
import pandas as pd

sample = flights[0]
bef  = pd.read_parquet(sample / "adsb_before.parquet")
adsc = pd.read_parquet(sample / "adsc.parquet")
aft  = pd.read_parquet(sample / "adsb_after.parquet")

print(f"ADS-B before : {len(bef):,} rows  ({bef['timestamp'].min()} → {bef['timestamp'].max()})")
print(f"ADS-C gap    : {len(adsc):,} rows  ({adsc['timestamp'].min()} → {adsc['timestamp'].max()})")
print(f"ADS-B after  : {len(aft):,} rows  ({aft['timestamp'].min()} → {aft['timestamp'].max()})")

t_gap_start = bef["timestamp"].max()
t_gap_end   = aft["timestamp"].min()
gap_min = (t_gap_end - t_gap_start).total_seconds() / 60
print(f"\nGap duration : {gap_min:.1f} minutes")
print(f"ADS-C points in gap: {len(adsc)}")
print()
print(bef[["timestamp","latitude","longitude","altitude"]].tail(3))

ADS-B before : 227 rows  (2023-08-10 06:00:00 → 2023-08-10 06:56:30)
ADS-C gap    : 444 rows  (2023-08-10 07:32:00 → 2023-08-10 09:22:45)
ADS-B after  : 199 rows  (2023-08-10 10:46:15 → 2023-08-10 11:35:45)

Gap duration : 229.8 minutes
ADS-C points in gap: 444

              timestamp   latitude  longitude  altitude
224 2023-08-10 06:56:00  54.458221  -14.15031    9753.6
225 2023-08-10 06:56:15  54.458221  -14.15031    9753.6
226 2023-08-10 06:56:30  54.458221  -14.15031    9753.6


## 1c. Dataset summary

In [4]:
import numpy as np

rows = []
for step_dir in sorted(CLEAN_DIR.iterdir()):
    if not step_dir.is_dir(): continue
    for flight_dir in sorted(step_dir.iterdir()):
        if not flight_dir.is_dir(): continue
        try:
            b = pd.read_parquet(flight_dir / "adsb_before.parquet")
            a = pd.read_parquet(flight_dir / "adsb_after.parquet")
            d = pd.read_parquet(flight_dir / "adsc.parquet")
            gap = (a["timestamp"].min() - b["timestamp"].max()).total_seconds() / 60
            rows.append({"flight": flight_dir.name, "step": step_dir.name,
                         "gap_min": round(gap,1), "adsc_pts": len(d),
                         "before_pts": len(b), "after_pts": len(a)})
        except: continue

summary = pd.DataFrame(rows)
print(f"Total flights : {len(summary)}")
print(f"Gap duration  : mean={summary['gap_min'].mean():.1f} min  "
      f"min={summary['gap_min'].min():.1f}  max={summary['gap_min'].max():.1f}")
print(f"ADS-C waypts  : mean={summary['adsc_pts'].mean():.1f}")
display(summary.head(10))

Total flights : 2146
Gap duration  : mean=242.9 min  min=31.2  max=604.2
ADS-C waypts  : mean=347.7


,flight,step,gap_min,adsc_pts,before_pts,after_pts
0,20230810_4ba959_073209_092245,step1_raw_2023-08-10_to_2023-09-10,229.8,444,227,199
1,20230811_7380c5_084005_094925,step1_raw_2023-08-10_to_2023-09-10,245.8,278,79,577
2,20230811_76cce1_093532_104531,step1_raw_2023-08-10_to_2023-09-10,193.0,281,235,638
3,20230811_89617a_100406_110809,step1_raw_2023-08-10_to_2023-09-10,244.0,257,628,94
4,20230812_06a12f_091555_104814,step1_raw_2023-08-10_to_2023-09-10,210.8,370,42,655
5,20230813_06a123_090843_103157,step1_raw_2023-08-10_to_2023-09-10,214.8,334,244,610
6,20230813_06a1be_090239_102345,step1_raw_2023-08-10_to_2023-09-10,210.8,326,225,678
7,20230813_7380c7_105912_120904,step1_raw_2023-08-10_to_2023-09-10,205.2,281,249,579
8,20230813_738101_064546_074907,step1_raw_2023-08-10_to_2023-09-10,218.2,254,92,193
9,20230813_8963e5_111404_121848,step1_raw_2023-08-10_to_2023-09-10,255.5,260,519,84
